# Custom-LLM V3 — 100M Decoder-Only Transformer

**Model:** Custom-LLM V3 · **Parameters:** 101,269,248 (verified in Section 6) · **Objective:** next-token prediction (causal LM, cross-entropy)

| Component | V3 value |
|---|---|
| Architecture | Decoder-only Transformer, pre-norm residual blocks |
| Layers / heads / width | 10 layers · 12 heads (head dim 64) · `d_model` 768 |
| Feed-forward | SwiGLU, hidden 2,304 |
| Normalization / positions | RMSNorm · RoPE |
| Attention | Causal self-attention via `F.scaled_dot_product_attention` |
| Embeddings | Tied input/output, vocabulary 32,000 |
| Context | 512 tokens |
| Optimizer / schedule | AdamW (β = 0.9, 0.95), warmup 1,000 steps then linear decay to 0.5× peak LR |
| Precision | FP16 autocast + GradScaler on CUDA (FP32 on CPU) |

**Data:** V3 ByteLevel BPE tokenizer (32K, full 256-byte alphabet) applied to FineWeb-Edu + English Wikipedia — 722,485 training and 14,473 validation sequences of 512 tokens (~370M training token positions).

**Hardware target:** Google Colab Tesla T4 (15 GB). The notebook also runs on any CUDA machine and, for validation and smoke tests only, on CPU.

**Source of truth** — this notebook only *imports and launches* project code:

| File | Role |
|---|---|
| [`training/train_v3.py`](training/train_v3.py) | Training loop, V3 dataset, evaluation, V3 checkpoint format |
| [`configs/100m_v3.yaml`](configs/100m_v3.yaml) | Architecture and training hyperparameters |
| [`model/`](model/model.py) | `TransformerLM` and its components |
| [`training/optimizer.py`](training/optimizer.py), [`training/scheduler.py`](training/scheduler.py) | AdamW and LR schedule |
| [`evaluation/evaluate.py`](evaluation/evaluate.py) | Full validation (FineWeb, Wikipedia, weighted combined) |
| [`inference/generate.py`](inference/generate.py), [`inference/sampling.py`](inference/sampling.py) | Text generation |
| [`data/scripts/tokenize.py`](data/scripts/tokenize.py) | Streaming JSONL → sharded `uint16` NumPy dataset |

**Safe to run top to bottom.** Nothing in this notebook trains, overwrites a checkpoint or deletes data unless you change one of the clearly marked flags (`TRAINING_MODE`, `CONFIRM_LAUNCH`, `RUN_FULL_VALIDATION`, `BACKUP_CHECKPOINT`).

## 2. Environment Setup

Detects Colab vs. local/server, mounts Google Drive on Colab, locates the repository, installs only *missing* packages, reports the hardware, and sets seeds.

Outside Colab, run Jupyter from the repository root or set `CUSTOM_LLM_REPO`.

In [ ]:
import os
import sys
import platform
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Jaydeep-Nadkarni/Custom-LLM.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/Custom-LLM")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path(os.environ.get("CUSTOM_LLM_REPO", Path.cwd())).resolve()

if not (REPO_ROOT / "training" / "train_v3.py").is_file():
    raise FileNotFoundError(
        f"{REPO_ROOT} is not a Custom-LLM checkout containing training/train_v3.py. "
        "Start Jupyter in the repository root, set CUSTOM_LLM_REPO, or push the "
        "latest repository files before cloning on Colab."
    )

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

commit = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print("Environment :", "Google Colab" if IN_COLAB else f"Local / server ({platform.system()})")
print("Repository  :", REPO_ROOT)
print("Git commit  :", commit or "unknown")

In [ ]:
import importlib.util

REQUIRED_PACKAGES = {"torch": "torch", "numpy": "numpy", "yaml": "pyyaml",
                     "tokenizers": "tokenizers", "matplotlib": "matplotlib"}
missing = [pip_name for module, pip_name in REQUIRED_PACKAGES.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("All required packages are already installed.")

In [ ]:
import random
import numpy as np
import torch
import yaml
import tokenizers

print(f"Python      : {platform.python_version()}")
print(f"PyTorch     : {torch.__version__}  (CUDA build: {torch.version.cuda})")
print(f"NumPy       : {np.__version__}")
print(f"tokenizers  : {tokenizers.__version__}")

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
GPU_NAME, GPU_MEMORY_GB, BF16_SUPPORTED, FLASH_SDPA = "none (CPU)", 0.0, False, False

if HAS_CUDA:
    props = torch.cuda.get_device_properties(0)
    GPU_NAME = props.name
    GPU_MEMORY_GB = props.total_memory / 1024**3
    BF16_SUPPORTED = torch.cuda.is_bf16_supported()
    try:
        from torch.nn.attention import SDPBackend, sdpa_kernel
        q = torch.randn(1, 2, 16, 64, device="cuda", dtype=torch.float16)
        with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
            torch.nn.functional.scaled_dot_product_attention(q, q, q, is_causal=True)
        FLASH_SDPA = True
    except Exception:
        FLASH_SDPA = False
    print(f"GPU         : {GPU_NAME}, {GPU_MEMORY_GB:.1f} GB, compute capability {props.major}.{props.minor}")
    print(f"cuDNN       : {torch.backends.cudnn.version()}")
else:
    print("GPU         : not available — smoke tests run on CPU, benchmarking and training are skipped")

print(f"BF16        : {'supported' if BF16_SUPPORTED else 'not supported'}")
print(f"Flash SDPA  : {'available' if FLASH_SDPA else 'not available (PyTorch picks memory-efficient or math SDPA)'}")
print("Note        : train_v3.py always trains with FP16 autocast + GradScaler on CUDA; BF16 is reported for information only.")

In [ ]:
SEED = 42  # same seed train_v3.py uses
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if HAS_CUDA:
    torch.cuda.manual_seed_all(SEED)

# Memory-mapped shards are shared by forked DataLoader workers on Linux. On Windows/macOS
# workers are spawned and the dataset is pickled, which copies the arrays, so use 0 there.
NUM_WORKERS = 2 if platform.system() == "Linux" else 0
print("Seed:", SEED, "| DataLoader workers:", NUM_WORKERS)

## 3. Repository and Configuration Validation

All run artefacts live in one **workspace** directory: Google Drive's `MyDrive/Custom-LLM-V2` on Colab, otherwise `<repo>/workspace` (override with `CUSTOM_LLM_WORKSPACE`). The layout matches the existing V3 run:

```
<workspace>/tokenizer/tokenizer_v3.json
<workspace>/tokenized_v3/{train,validation}/{inputs,targets,metadata}/
<workspace>/checkpoints_v3/100m_v3_latest.pt
<workspace>/logs_v3/100m_v3_training.log
```

Missing paths raise an error; nothing falls back to a different location.

In [ ]:
DEFAULT_WORKSPACE = Path("/content/drive/MyDrive/Custom-LLM-V2") if IN_COLAB else REPO_ROOT / "workspace"
WORKSPACE = Path(os.environ.get("CUSTOM_LLM_WORKSPACE", DEFAULT_WORKSPACE)).resolve()

CONFIG_PATH = REPO_ROOT / "configs" / "100m_v3.yaml"
TOKENIZER_PATH = WORKSPACE / "tokenizer" / "tokenizer_v3.json"
DATA_ROOT = WORKSPACE / "tokenized_v3"
CHECKPOINT_DIR = WORKSPACE / "checkpoints_v3"
LOG_DIR = WORKSPACE / "logs_v3"
SMOKE_DIR = WORKSPACE / "smoke_tests"

PRODUCTION_CHECKPOINT = CHECKPOINT_DIR / "100m_v3_latest.pt"
PRODUCTION_LOG = LOG_DIR / "100m_v3_training.log"

REQUIRED_SOURCE_FILES = [
    "training/train_v3.py", "training/optimizer.py", "training/scheduler.py",
    "model/model.py", "model/attention.py", "model/rope.py", "model/rmsnorm.py",
    "model/swiglu.py", "model/transformer_block.py", "model/embeddings.py",
    "evaluation/evaluate.py", "inference/generate.py", "inference/sampling.py",
    "configs/100m_v3.yaml",
]
missing_sources = [f for f in REQUIRED_SOURCE_FILES if not (REPO_ROOT / f).is_file()]
if missing_sources:
    raise FileNotFoundError(f"Missing project files: {missing_sources}")

print("Workspace :", WORKSPACE)
print("All", len(REQUIRED_SOURCE_FILES), "required source files found.")

In [ ]:
REQUIRED_KEYS = {
    "model": ["vocab_size", "d_model", "n_heads", "n_layers", "hidden_dim", "context_length", "dropout"],
    "training": ["batch_size", "steps", "learning_rate", "weight_decay", "warmup_steps",
                 "log_every", "checkpoint_every", "eval_every", "eval_samples"],
    "data": ["train_inputs", "train_targets", "val_inputs", "val_targets"],
}

config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
for section, keys in REQUIRED_KEYS.items():
    absent = [k for k in keys if k not in config.get(section, {})]
    if absent:
        raise KeyError(f"{CONFIG_PATH.name}: missing {section} keys {absent}")

# train_v3.py reads data paths from the config, so point them at this workspace.
resolved_data = {
    "train_inputs": str(DATA_ROOT / "train" / "inputs"),
    "train_targets": str(DATA_ROOT / "train" / "targets"),
    "val_inputs": str(DATA_ROOT / "validation" / "inputs"),
    "val_targets": str(DATA_ROOT / "validation" / "targets"),
}
changed = {k: (config["data"][k], v) for k, v in resolved_data.items()
           if Path(config["data"][k]) != Path(v)}

if changed:
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    config["data"].update(resolved_data)
    RUN_CONFIG_PATH = LOG_DIR / "100m_v3_runtime.yaml"
    RUN_CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    print("Data paths differ from the committed config; wrote a runtime copy (only data paths changed):")
    for key, (old, new) in changed.items():
        print(f"  {key}: {old}  ->  {new}")
else:
    RUN_CONFIG_PATH = CONFIG_PATH

print("\nConfig used for training/evaluation:", RUN_CONFIG_PATH)
print(yaml.safe_dump(config, sort_keys=False))

In [ ]:
def require(path, what, hint=""):
    if not Path(path).exists():
        raise FileNotFoundError(f"{what} not found: {path}" + (f"\n{hint}" if hint else ""))
    return Path(path)

require(TOKENIZER_PATH, "V3 tokenizer",
        "Copy tokenizer_v3.json (the 256-byte-alphabet tokenizer) to this location.")
for key, path in resolved_data.items():
    require(path, f"Dataset directory '{key}'",
            "Generate it with data/scripts/tokenize.py (see Appendix B) or fix WORKSPACE.")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Tokenizer             : {TOKENIZER_PATH}")
print(f"Dataset root          : {DATA_ROOT}")
print(f"Checkpoint directory  : {CHECKPOINT_DIR}")
print(f"Production checkpoint : {PRODUCTION_CHECKPOINT} ({'EXISTS' if PRODUCTION_CHECKPOINT.exists() else 'not created yet'})")
print(f"Production log        : {PRODUCTION_LOG} ({'EXISTS' if PRODUCTION_LOG.exists() else 'not created yet'})")

In [ ]:
V3_MODEL = {"vocab_size": 32000, "d_model": 768, "n_heads": 12, "n_layers": 10,
            "hidden_dim": 2304, "context_length": 512, "dropout": 0.0}
V3_PARAMETERS = 101_269_248

mc = config["model"]
if mc["d_model"] % mc["n_heads"]:
    raise ValueError("d_model must be divisible by n_heads")
if (mc["d_model"] // mc["n_heads"]) % 2:
    raise ValueError("RoPE needs an even head dimension")
if mc != V3_MODEL:
    diff = {k: (mc.get(k), v) for k, v in V3_MODEL.items() if mc.get(k) != v}
    raise ValueError(f"Config is not the V3 100M architecture (config, expected): {diff}")

# Analytical count for this implementation: tied embedding + per-layer (QKV, output, SwiGLU, 2 norms) + final norm.
d, f, L, V = mc["d_model"], mc["hidden_dim"], mc["n_layers"], mc["vocab_size"]
EXPECTED_PARAMETERS = V * d + L * (4 * d * d + 3 * d * f + 2 * d) + d
assert EXPECTED_PARAMETERS == V3_PARAMETERS, EXPECTED_PARAMETERS
print(f"Architecture matches V3; analytical parameter count {EXPECTED_PARAMETERS:,}")

## 4. Tokenizer Validation

Read-only checks of the V3 tokenizer — nothing here trains or overwrites a tokenizer.

In [ ]:
import hashlib
from tokenizers import Tokenizer
from tokenizers.pre_tokenizers import ByteLevel

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
TOKENIZER_SHA256 = hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest()
VOCAB_SIZE = tokenizer.get_vocab_size()

SPECIAL_TOKENS = {"<PAD>": 0, "<UNK>": 1, "<BOS>": 2, "<EOS>": 3}
special_ids = {t: tokenizer.token_to_id(t) for t in SPECIAL_TOKENS}
UNK_ID, EOS_ID = special_ids["<UNK>"], special_ids["<EOS>"]
missing_bytes = [c for c in ByteLevel.alphabet() if tokenizer.token_to_id(c) is None]

print("SHA-256          :", TOKENIZER_SHA256)
print("Vocabulary size  :", VOCAB_SIZE)
print("Special tokens   :", special_ids)
print("ByteLevel bytes  :", 256 - len(missing_bytes), "/ 256 present")

assert VOCAB_SIZE == config["model"]["vocab_size"], "Tokenizer and model vocabulary differ"
assert special_ids == SPECIAL_TOKENS, "Special-token IDs differ from the V3 tokenizer"
assert not missing_bytes, (
    f"{len(missing_bytes)} ByteLevel characters are missing — this is the V1 tokenizer, which produces UNKs. Use tokenizer_v3.json."
)

In [ ]:
ROUND_TRIP_TEXTS = [
    "Artificial intelligence is transforming technology.",
    "First paragraph.\n\nSecond paragraph.",
    "Hello world! This is a simple test.",
    "The capital of France is Paris.",
    "Machine learning, deep learning, and neural networks.",
    "भारत is a country in South Asia.",
    "Special characters: @#$%^&*()_+=[]{}",
    "Tabs\tand   multiple   spaces, emoji 🚀, accents: café naïve Ångström.",
]

failures = []
for text in ROUND_TRIP_TEXTS:
    ids = tokenizer.encode(text).ids
    decoded = tokenizer.decode(ids)
    unk = ids.count(UNK_ID)
    in_range = all(0 <= i < VOCAB_SIZE for i in ids)
    ok = decoded == text and unk == 0 and in_range
    failures += [] if ok else [text]
    print(f"{'PASS' if ok else 'FAIL'}  tokens={len(ids):3d}  unk={unk}  {text!r}")

print("\nSpecial tokens are atomic:",
      {t: tokenizer.encode(t, add_special_tokens=False).ids for t in SPECIAL_TOKENS})
assert not failures, f"Round-trip failures: {failures}"
print("Tokenizer validation PASSED")

## 5. Dataset Inspection

Shards are opened memory-mapped through the project's own `V3TokenDataset`, so nothing is loaded into RAM in full. Checks cover:
- input/target shard pairing, plus agreement with the metadata files;
- dtype, sequence length and token-ID range;
- next-token alignment, including continuity across rows and shards;
- a sampled check for train/validation leakage.

In [ ]:
import json
from training.train_v3 import V3TokenDataset

SEQ_LEN = config["model"]["context_length"]
V3_REFERENCE_COUNTS = {"train": 722_485, "fineweb_validation": 9_997, "wikipedia_validation": 4_476}

shard_info = {}
for split in ("train", "validation"):
    split_dir = DATA_ROOT / split
    inputs = sorted((split_dir / "inputs").glob("*.npy"))
    targets = sorted((split_dir / "targets").glob("*.npy"))
    if [p.name for p in inputs] != [p.name for p in targets]:
        raise ValueError(f"{split}: input and target shard names differ")

    rows = {p.name: np.load(p, mmap_mode="r").shape[0] for p in inputs}
    metadata = [json.loads(p.read_text()) for p in sorted((split_dir / "metadata").glob("*.json"))]
    meta_sequences = sum(m["sequence_count"] for m in metadata)
    meta_shards = sum(m["shard_count"] for m in metadata)

    for m in metadata:
        assert m["vocab_size"] == VOCAB_SIZE, f"{m['source']}: vocab {m['vocab_size']} != tokenizer {VOCAB_SIZE}"
        assert m["eos_id"] == EOS_ID, f"{m['source']}: eos_id {m['eos_id']} != {EOS_ID}"
        assert m["block_size"] == SEQ_LEN, f"{m['source']}: block_size {m['block_size']} != {SEQ_LEN}"
        assert m["dtype"] == "uint16"
    assert sum(rows.values()) == meta_sequences, f"{split}: shard rows {sum(rows.values())} != metadata {meta_sequences}"
    assert len(inputs) == meta_shards, f"{split}: {len(inputs)} shards != metadata {meta_shards}"

    shard_info[split] = {"inputs": inputs, "targets": targets, "metadata": metadata}
    print(f"{split.upper():10s} shards={len(inputs):3d}  sequences={sum(rows.values()):>9,}  "
          f"stream tokens={sum(m['total_tokens'] for m in metadata):>13,}")
    for m in metadata:
        print(f"    {Path(m['source']).name:28s} docs={m.get('documents', 'n/a'):>8}  seqs={m['sequence_count']:>8,}  shards={m['shard_count']}")
print("Shard pairing and metadata checks PASSED")

In [ ]:
# Exactly the datasets train_v3.py builds.
val_root = DATA_ROOT / "validation"
datasets = {
    "train": V3TokenDataset(DATA_ROOT / "train/inputs", DATA_ROOT / "train/targets", SEQ_LEN),
    "fineweb_validation": V3TokenDataset(val_root / "inputs/fineweb_validation_00000.npy",
                                         val_root / "targets/fineweb_validation_00000.npy", SEQ_LEN),
    "wikipedia_validation": V3TokenDataset(val_root / "inputs/wikipedia_validation_00000.npy",
                                           val_root / "targets/wikipedia_validation_00000.npy", SEQ_LEN),
}
DATASET_COUNTS = {name: len(ds) for name, ds in datasets.items()}
TRAIN_TOKEN_POSITIONS = DATASET_COUNTS["train"] * SEQ_LEN

for name, count in DATASET_COUNTS.items():
    ref = V3_REFERENCE_COUNTS[name]
    print(f"{name:22s} {count:>9,} sequences  ({'matches V3 reference' if count == ref else f'V3 reference is {ref:,}'})")
print(f"Training token positions: {TRAIN_TOKEN_POSITIONS:,}")

x, y = datasets["train"][0]
assert x.shape == y.shape == (SEQ_LEN,) and x.dtype == y.dtype == torch.int64

In [ ]:
FULL_SCAN = False   # True = scan every training shard (slow on Google Drive); validation shards are always scanned

def scan_shard(input_path, target_path):
    x = np.load(input_path, mmap_mode="r")
    y = np.load(target_path, mmap_mode="r")
    assert x.dtype == y.dtype == np.uint16, f"{input_path.name}: dtype {x.dtype}/{y.dtype}"
    assert x.shape == y.shape and x.ndim == 2 and x.shape[1] == SEQ_LEN, f"{input_path.name}: shape {x.shape}"
    assert int(x.max()) < VOCAB_SIZE and int(y.max()) < VOCAB_SIZE, f"{input_path.name}: token ID out of range"
    assert np.array_equal(x[:, 1:], y[:, :-1]), f"{input_path.name}: targets are not inputs shifted by one"
    assert np.array_equal(x[1:, 0], y[:-1, -1]), f"{input_path.name}: rows are not a contiguous token stream"
    return int(np.count_nonzero(x == UNK_ID)), x[0, 0], y[-1, -1]

rng = np.random.default_rng(SEED)
total_unk = 0
for split, info in shard_info.items():
    pairs = list(zip(info["inputs"], info["targets"]))
    if split == "train" and not FULL_SCAN and len(pairs) > 5:
        chosen = sorted({0, len(pairs) - 1, *rng.choice(len(pairs), 3, replace=False).tolist()})
    else:
        chosen = range(len(pairs))
    edges = {}
    for i in chosen:
        unk, first, last = scan_shard(*pairs[i])
        total_unk += unk
        edges[i] = (pairs[i][0].name.rsplit("_", 1)[0], first, last)
    # Consecutive shards of the same source continue the same token stream.
    for i in chosen:
        if i + 1 in edges and edges[i][0] == edges[i + 1][0]:
            assert edges[i][2] == edges[i + 1][1], f"Stream break between shards {i} and {i + 1}"
    print(f"{split:10s}: scanned {len(list(chosen))}/{len(pairs)} shards — dtype, shape, range, alignment, continuity OK")

print("UNK tokens in scanned inputs:", total_unk)
assert total_unk == 0, "UNK tokens found — data was not produced with the V3 tokenizer"

In [ ]:
LEAKAGE_SAMPLE = 20_000   # random training sequences compared against every validation sequence

val_hashes = set()
for name in ("fineweb_validation", "wikipedia_validation"):
    arr = datasets[name].inputs[0]
    val_hashes.update(hashlib.blake2b(row.tobytes(), digest_size=16).digest() for row in arr)

train_ds = datasets["train"]
sample = rng.choice(len(train_ds), min(LEAKAGE_SAMPLE, len(train_ds)), replace=False)
overlaps = sum(
    hashlib.blake2b(train_ds[int(i)][0].numpy().astype(np.uint16).tobytes(), digest_size=16).digest() in val_hashes
    for i in sample
)
train_names = {p.name for p in shard_info["train"]["inputs"]}
val_names = {p.name for p in shard_info["validation"]["inputs"]}
train_sources = {m["source"] for m in shard_info["train"]["metadata"]}
val_sources = {m["source"] for m in shard_info["validation"]["metadata"]}

print(f"Sampled leakage check: {overlaps} of {len(sample):,} training sequences appear in validation")
print("Shard filename overlap:", train_names & val_names or "none")
print("Source file overlap   :", train_sources & val_sources or "none")
assert overlaps == 0 and not (train_names & val_names) and not (train_sources & val_sources)

In [ ]:
for name, ds in datasets.items():
    x, y = ds[int(rng.integers(len(ds)))]
    print(f"--- {name} ---")
    print("input IDs :", x[:12].tolist())
    print("target IDs:", y[:12].tolist())
    print(repr(tokenizer.decode(x.tolist(), skip_special_tokens=False)[:300]), "\n")

## 6. Model Initialization

The model is built with the same constructor arguments `train_v3.py` uses (`inference.generate.build_model_from_config`).

In [ ]:
from inference.generate import build_model_from_config

torch.manual_seed(SEED)
model = build_model_from_config(config["model"])

PARAM_COUNT = sum(p.numel() for p in model.parameters())
TRAINABLE_COUNT = sum(p.numel() for p in model.parameters() if p.requires_grad)
breakdown = {
    "token embedding (tied)": model.embedding.embedding.weight.numel(),
    "attention (QKV + output)": sum(p.numel() for n, p in model.named_parameters() if ".attention." in n),
    "SwiGLU feed-forward": sum(p.numel() for n, p in model.named_parameters() if ".ffn." in n),
    "RMSNorm weights": sum(p.numel() for n, p in model.named_parameters() if "norm" in n),
}

print(model.layers[0])
print(f"\nLayers {len(model.layers)} | d_model {model.d_model} | heads {model.layers[0].attention.n_heads} "
      f"| head dim {model.layers[0].attention.head_dim} | FFN {config['model']['hidden_dim']} | context {model.max_seq_len}")
for part, n in breakdown.items():
    print(f"  {part:26s} {n:>12,}")
print(f"Total parameters     : {PARAM_COUNT:,}")
print(f"Trainable parameters : {TRAINABLE_COUNT:,}")
assert PARAM_COUNT == EXPECTED_PARAMETERS == V3_PARAMETERS

gb = lambda n_bytes: n_bytes / 1024**3
print("\nStatic training memory (FP32 master weights + AdamW):")
print(f"  weights {gb(4 * PARAM_COUNT):.2f} GB + gradients {gb(4 * PARAM_COUNT):.2f} GB "
      f"+ Adam moments {gb(8 * PARAM_COUNT):.2f} GB = {gb(16 * PARAM_COUNT):.2f} GB  (activations measured in Sections 7-8)")

## 7. Model Smoke Tests

Real forward, backward and optimizer tests on a real training batch. They use the same precision path as `train_v3.py`: FP16 autocast + GradScaler on CUDA, FP32 on CPU. On CPU the sequence length is shortened to keep the test quick.

In [ ]:
import torch.nn.functional as F
from training.optimizer import build_optimizer
from training.scheduler import build_scheduler

SMOKE_BATCH = 2
SMOKE_SEQ = SEQ_LEN if HAS_CUDA else 128
USE_AMP = HAS_CUDA

model = model.to(DEVICE)
batch = [datasets["train"][i] for i in range(SMOKE_BATCH)]
inputs = torch.stack([b[0] for b in batch])[:, :SMOKE_SEQ].to(DEVICE)
targets = torch.stack([b[1] for b in batch])[:, :SMOKE_SEQ].to(DEVICE)

# 1. Forward shape + causality (FP32, eval mode): changing later tokens must not change earlier logits.
model.eval()
with torch.no_grad():
    logits_a = model(inputs)
    perturbed = inputs.clone()
    half = SMOKE_SEQ // 2
    perturbed[:, half:] = torch.randint(4, VOCAB_SIZE, perturbed[:, half:].shape, device=DEVICE)
    logits_b = model(perturbed)
assert logits_a.shape == (SMOKE_BATCH, SMOKE_SEQ, VOCAB_SIZE), logits_a.shape
assert torch.allclose(logits_a[:, :half], logits_b[:, :half], atol=1e-3), "Future tokens leak into past positions"
assert not torch.allclose(logits_a[:, half:], logits_b[:, half:]), "Perturbation had no effect"
print("Forward shape", tuple(logits_a.shape), "| causal masking verified")
del logits_a, logits_b

In [ ]:
model.train()
optimizer = build_optimizer(model, config["training"]["learning_rate"], config["training"]["weight_decay"])
scheduler = build_scheduler(optimizer, config["training"]["warmup_steps"], config["training"]["steps"])
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
before = model.layers[0].ffn.down_proj.weight.detach().clone()

if HAS_CUDA:
    torch.cuda.reset_peak_memory_stats()
optimizer.zero_grad(set_to_none=True)
with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
    logits = model(inputs)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))

assert torch.isfinite(loss), loss
print(f"Initial loss {loss.item():.4f} (uniform-prediction reference ln(32000) = {np.log(VOCAB_SIZE):.4f})")
assert 9.0 < loss.item() < 12.0, "Initial loss far from ln(vocab) — check initialization or targets"

scaler.scale(loss).backward()
scaler.unscale_(optimizer)
bad = [n for n, p in model.named_parameters() if p.grad is None or not torch.isfinite(p.grad).all()]
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
assert not bad, f"Missing or non-finite gradients: {bad[:5]}"

scaler.step(optimizer)
scaler.update()
scheduler.step()
delta = (model.layers[0].ffn.down_proj.weight - before).abs().max().item()
assert delta > 0, "Optimizer step did not change the weights"

with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
    loss_after = F.cross_entropy(model(inputs).reshape(-1, VOCAB_SIZE), targets.reshape(-1))

print(f"Gradients finite for all {sum(1 for _ in model.parameters())} tensors | grad norm {float(grad_norm):.4f}")
print(f"Optimizer step changed weights (max |Δ| {delta:.2e}) | LR now {optimizer.param_groups[0]['lr']:.2e}")
print(f"Loss after one step {loss_after.item():.4f} | precision {'FP16 autocast + GradScaler' if USE_AMP else 'FP32'}")
if HAS_CUDA:
    print(f"Peak GPU memory (batch {SMOKE_BATCH} x {SMOKE_SEQ}): {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("SMOKE TESTS PASSED")

del optimizer, scheduler, scaler, logits, loss, loss_after, before
model = model.cpu()
if HAS_CUDA:
    torch.cuda.empty_cache()

## 8. GPU Memory and Batch-Size Benchmark

Times complete training steps (FP16 autocast, GradScaler, gradient clipping, AdamW) at full 512-token context with synthetic tokens. Memory and speed don't depend on token values. Batch sizes increase until the first out-of-memory error. Needs CUDA; skipped on CPU.

`train_v3.py` has **no gradient accumulation**: the effective batch is `batch_size`. Changing `batch_size` while resuming an existing run changes that experiment, so use these numbers for new runs.

In [ ]:
BENCHMARK_BATCH_SIZES = [4, 8, 16, 24, 32]
BENCHMARK_STEPS = 3
BENCHMARK_RESULTS = []

def timed(fn):
    start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    start.record(); out = fn(); end.record(); torch.cuda.synchronize()
    return out, start.elapsed_time(end) / 1000

if not HAS_CUDA:
    print("CUDA not available — benchmark skipped.")
else:
    bench_model = build_model_from_config(config["model"]).to(DEVICE).train()
    bench_opt = build_optimizer(bench_model, config["training"]["learning_rate"], config["training"]["weight_decay"])
    bench_scaler = torch.amp.GradScaler("cuda")
    for bs in BENCHMARK_BATCH_SIZES:
        try:
            x = torch.randint(4, VOCAB_SIZE, (bs, SEQ_LEN), device=DEVICE)
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            times = []
            for step in range(BENCHMARK_STEPS + 1):          # first step is warm-up
                bench_opt.zero_grad(set_to_none=True)
                def fwd():
                    with torch.autocast("cuda", dtype=torch.float16):
                        return F.cross_entropy(bench_model(x).reshape(-1, VOCAB_SIZE), x.reshape(-1))
                loss, t_fwd = timed(fwd)
                _, t_bwd = timed(lambda: bench_scaler.scale(loss).backward())
                def opt_step():
                    bench_scaler.unscale_(bench_opt)
                    torch.nn.utils.clip_grad_norm_(bench_model.parameters(), 1.0)
                    bench_scaler.step(bench_opt); bench_scaler.update()
                _, t_opt = timed(opt_step)
                if step:
                    times.append((t_fwd, t_bwd, t_opt))
            f_, b_, o_ = (float(np.mean(c)) for c in zip(*times))
            peak = torch.cuda.max_memory_allocated() / 1024**3
            BENCHMARK_RESULTS.append({"batch": bs, "peak_gb": peak, "fwd_s": f_, "bwd_s": b_, "opt_s": o_,
                                      "tokens_per_s": bs * SEQ_LEN / (f_ + b_ + o_)})
            del x, loss
        except torch.cuda.OutOfMemoryError:
            print(f"batch {bs}: out of memory — stopping")
            break
    del bench_model, bench_opt, bench_scaler
    torch.cuda.empty_cache()

    print(f"{'batch':>5} {'peak GB':>8} {'fwd s':>7} {'bwd s':>7} {'opt s':>7} {'tokens/s':>10}")
    for r in BENCHMARK_RESULTS:
        print(f"{r['batch']:>5} {r['peak_gb']:>8.2f} {r['fwd_s']:>7.3f} {r['bwd_s']:>7.3f} {r['opt_s']:>7.3f} {r['tokens_per_s']:>10,.0f}")
    safe = [r for r in BENCHMARK_RESULTS if r["peak_gb"] < 0.85 * GPU_MEMORY_GB]
    if safe:
        best = max(safe, key=lambda r: r["batch"])
        print(f"\nLargest batch under 85% of {GPU_MEMORY_GB:.1f} GB: {best['batch']} "
              f"({best['peak_gb']:.2f} GB, {best['tokens_per_s']:,.0f} tokens/s). "
              f"Configured batch_size: {config['training']['batch_size']}.")

## 9. Training Configuration

Everything below is read from the YAML that the launch cell passes to `train_v3.py`. The launch command adds only paths and the DataLoader worker count. No hyperparameter is overridden from the notebook.

In [ ]:
tc = config["training"]
for key in ("batch_size", "steps", "warmup_steps", "log_every", "checkpoint_every", "eval_every", "eval_samples"):
    assert int(tc[key]) > 0, f"training.{key} must be positive"
assert tc["warmup_steps"] < tc["steps"], "warmup must be shorter than training"

tokens_per_step = tc["batch_size"] * SEQ_LEN
total_tokens = tokens_per_step * tc["steps"]

# Actual LR values from the project's scheduler (not a re-implementation).
probe = torch.optim.SGD([torch.nn.Parameter(torch.zeros(1))], lr=tc["learning_rate"])
probe_sched = build_scheduler(probe, tc["warmup_steps"], tc["steps"])
lr_at = {}
for step in range(1, tc["steps"] + 1):
    probe.step(); probe_sched.step()
    if step in {1, tc["warmup_steps"], tc["steps"] // 2, tc["steps"]}:
        lr_at[step] = probe.param_groups[0]["lr"]

rows = [
    ("Batch size (micro = global)", tc["batch_size"]),
    ("Gradient accumulation", "1 (not supported by train_v3.py)"),
    ("Context length", SEQ_LEN),
    ("Tokens per optimizer step", f"{tokens_per_step:,}"),
    ("Optimizer steps", f"{tc['steps']:,}"),
    ("Total training tokens", f"{total_tokens:,}  ({total_tokens / TRAIN_TOKEN_POSITIONS:.2f} epochs of the training set)"),
    ("Peak learning rate", tc["learning_rate"]),
    ("Warmup steps", tc["warmup_steps"]),
    ("LR schedule (actual)", ", ".join(f"step {s}: {lr:.2e}" for s, lr in lr_at.items())),
    ("Weight decay", tc["weight_decay"]),
    ("Gradient clipping", "1.0 (fixed in train_v3.py)"),
    ("Precision", "FP16 autocast + GradScaler" if HAS_CUDA else "FP32 (CPU)"),
    ("Checkpoint interval", tc["checkpoint_every"]),
    ("Sampled-eval interval / samples per source", f"{tc['eval_every']} / {tc['eval_samples']}"),
    ("Full-eval interval", tc.get("full_eval_every", 0) or "final step only"),
    ("DataLoader workers", NUM_WORKERS),
]
for name, value in rows:
    print(f"{name:44s} {value}")

## 10. Training Launch

Training runs `python -m training.train_v3` as a subprocess. Its loop isn't duplicated here.

| `TRAINING_MODE` | What happens | Writes to |
|---|---|---|
| `"none"` (default) | Prints the command only | nothing |
| `"smoke"` | 3 steps, batch 2, 4 eval samples — checks the CLI end to end | `<workspace>/smoke_tests/` (replaced each time) |
| `"fresh"` | New run from random initialization | production checkpoint + log; **refuses if either already exists** |
| `"resume"` | Continues from `RESUME_FROM` after a compatibility check | production checkpoint (atomic replace) + appends to log |

`"fresh"` and `"resume"` also require `CONFIRM_LAUNCH = True`. Back up the checkpoint first (Section 13) if you want to keep the current state.

In [ ]:
from training.train_v3 import CHECKPOINT_FORMAT

def load_compatible_checkpoint(path, require_optimizer=True):
    # Load a checkpoint (memory-mapped) and refuse anything that is not V3 with this exact architecture.
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    state = torch.load(path, map_location="cpu", weights_only=False, mmap=True)
    if state.get("format") != CHECKPOINT_FORMAT:
        raise ValueError(f"{path.name}: format {state.get('format')!r} is not {CHECKPOINT_FORMAT!r} "
                         "(V1/V2 checkpoints use a different tokenizer and cannot be used for V3)")
    if state["config"]["model"] != config["model"]:
        raise ValueError(f"{path.name}: architecture {state['config']['model']} differs from the config")
    if require_optimizer and not all(k in state for k in ("optimizer", "scheduler", "scaler")):
        raise ValueError(f"{path.name}: missing optimizer/scheduler/scaler state, cannot resume")
    return state

In [ ]:
TRAINING_MODE = "none"            # "none" | "smoke" | "fresh" | "resume"
CONFIRM_LAUNCH = False            # must be True for "fresh" and "resume"
RESUME_FROM = PRODUCTION_CHECKPOINT

In [ ]:
import shutil

def build_command(mode):
    cmd = [sys.executable, "-m", "training.train_v3", "--config", str(RUN_CONFIG_PATH),
           "--num-workers", str(NUM_WORKERS)]
    if mode == "smoke":
        return cmd + ["--steps", "3", "--batch-size", "2", "--log-every", "1", "--checkpoint-every", "3",
                      "--eval-every", "3", "--eval-samples", "4",
                      "--checkpoint-path", str(SMOKE_DIR / "v3_smoke.pt"),
                      "--log-file", str(SMOKE_DIR / "v3_smoke.log")]
    cmd += ["--checkpoint-path", str(PRODUCTION_CHECKPOINT), "--log-file", str(PRODUCTION_LOG)]
    if mode == "resume":
        cmd += ["--resume-from", str(RESUME_FROM)]
    return cmd

def preflight(mode):
    if mode == "smoke":
        SMOKE_DIR.mkdir(parents=True, exist_ok=True)
        for p in (SMOKE_DIR / "v3_smoke.pt", SMOKE_DIR / "v3_smoke.log"):
            p.unlink(missing_ok=True)
    elif mode == "fresh":
        for p in (PRODUCTION_CHECKPOINT, PRODUCTION_LOG):
            if p.exists():
                raise FileExistsError(f"{p} already exists. Resume instead, or move it away after checking it.")
    elif mode == "resume":
        state = load_compatible_checkpoint(RESUME_FROM)
        if state["step"] >= config["training"]["steps"]:
            raise ValueError(f"Checkpoint is at step {state['step']}, already at training.steps={config['training']['steps']}")
        print(f"Resume checkpoint OK: step {state['step']} of {config['training']['steps']}")
        del state
    print(f"Free disk at checkpoint location: {shutil.disk_usage(CHECKPOINT_DIR).free / 1024**3:.1f} GB")

def run(cmd):
    print("$", " ".join(cmd), "\n")
    process = subprocess.Popen(cmd, cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"train_v3.py exited with code {process.returncode}")

if TRAINING_MODE not in {"none", "smoke", "fresh", "resume"}:
    raise ValueError(f"Unknown TRAINING_MODE {TRAINING_MODE!r}")
if TRAINING_MODE == "none":
    for mode in ("smoke", "fresh", "resume"):
        print(f"[{mode}]", " ".join(build_command(mode)), "\n")
    print("TRAINING_MODE is 'none' — nothing launched.")
elif TRAINING_MODE in {"fresh", "resume"} and not CONFIRM_LAUNCH:
    print(" ".join(build_command(TRAINING_MODE)))
    print(f"\nSet CONFIRM_LAUNCH = True to start a {TRAINING_MODE} production run.")
else:
    preflight(TRAINING_MODE)
    run(build_command(TRAINING_MODE))

## 11. Training Monitoring

Parses the log that `train_v3.py` writes. Every value comes from that log; nothing is estimated. `train_v3.py` doesn't log timestamps, throughput or GPU usage, so those appear as "not logged". Live GPU state is read from `nvidia-smi` when it's available.

In [ ]:
import re

TRAIN_RE = re.compile(r"^step=(\d+)/(\d+) train_loss=([\d.]+) lr=([\d.eE+-]+) grad_norm=(\S+)"
                      r"(?: scale=([\d.]+) optimizer_skipped=(True|False))?")
VAL_RE = re.compile(r"^(FULL_)?VALIDATION step=(\d+) (fineweb|wikipedia|combined_weighted)_loss=([\d.]+)")
CKPT_RE = re.compile(r"^CHECKPOINT saved step=(\d+) path=(.+)$")

def parse_training_log(path):
    log = {"train": [], "val": {}, "full_val": {}, "checkpoints": []}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if m := TRAIN_RE.match(line):
            log["train"].append({"step": int(m[1]), "total": int(m[2]), "loss": float(m[3]), "lr": float(m[4]),
                                 "grad_norm": float(m[5]), "scale": float(m[6]) if m[6] else None,
                                 "skipped": m[7] == "True" if m[7] else None})
        elif m := VAL_RE.match(line):
            log["full_val" if m[1] else "val"].setdefault(int(m[2]), {})[m[3]] = float(m[4])
        elif m := CKPT_RE.match(line):
            log["checkpoints"].append({"step": int(m[1]), "path": m[2]})
    return log

smoke_log = SMOKE_DIR / "v3_smoke.log"
MONITOR_LOG = PRODUCTION_LOG if PRODUCTION_LOG.exists() else (smoke_log if smoke_log.exists() else None)
LOG = parse_training_log(MONITOR_LOG) if MONITOR_LOG else None

if LOG is None or not LOG["train"]:
    print("No training log yet.")
else:
    last = LOG["train"][-1]
    print(f"Log            : {MONITOR_LOG}{'  (SMOKE TEST)' if MONITOR_LOG == smoke_log else ''}")
    print(f"Step           : {last['step']:,} / {last['total']:,}  ({100 * last['step'] / last['total']:.1f}%)")
    print(f"Train loss     : {last['loss']:.4f}   (average over the last log window)")
    print(f"Learning rate  : {last['lr']:.3e}")
    print(f"Gradient norm  : {last['grad_norm']:.4f}  (pre-clipping)")
    print(f"GradScaler     : scale={last['scale']}  skipped_last_step={last['skipped']}")
    skipped = sum(1 for r in LOG["train"] if r["skipped"])
    print(f"Skipped steps  : {skipped} of {len(LOG['train'])} logged steps")
    if LOG["val"]:
        s = max(LOG["val"]); v = LOG["val"][s]
        print(f"Sampled val    : step {s}: " + ", ".join(f"{k} {l:.4f} (ppl {np.exp(min(l, 20)):.1f})" for k, l in v.items()))
    if LOG["checkpoints"]:
        c = LOG["checkpoints"][-1]
        print(f"Last checkpoint: step {c['step']} -> {c['path']}")
    print("Throughput     : not logged by train_v3.py")

if shutil.which("nvidia-smi"):
    q = "name,utilization.gpu,memory.used,memory.total,temperature.gpu"
    print("\nGPU now:", subprocess.run(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader"],
                                     capture_output=True, text=True).stdout.strip())

In [ ]:
import matplotlib.pyplot as plt

if LOG and LOG["train"]:
    steps = [r["step"] for r in LOG["train"]]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(steps, [r["loss"] for r in LOG["train"]], label="train (window mean)")
    for name, style in (("fineweb", "o-"), ("wikipedia", "s-"), ("combined_weighted", "^-")):
        pts = [(s, v[name]) for s, v in sorted(LOG["val"].items()) if name in v]
        if pts:
            axes[0].plot(*zip(*pts), style, ms=3, label=f"val {name} (sampled)")
    for s, v in sorted(LOG["full_val"].items()):
        if "combined_weighted" in v:
            axes[0].plot(s, v["combined_weighted"], "k*", ms=12, label="full val combined")
    axes[0].set(title="Loss", xlabel="step", ylabel="cross-entropy"); axes[0].legend(fontsize=8)
    axes[1].plot(steps, [r["lr"] for r in LOG["train"]]); axes[1].set(title="Learning rate", xlabel="step")
    axes[2].plot(steps, [r["grad_norm"] for r in LOG["train"]]); axes[2].set(title="Gradient norm (pre-clip)", xlabel="step")
    plt.tight_layout(); plt.show()
else:
    print("Nothing to plot yet.")

## 12. Validation and Evaluation

- **Sampled validation** happens during training: `eval_samples` fixed sequences per source (seeds 1234 / 5678). The combined loss weights each source by its full validation size. Use it for curves; it isn't a final metric.
- **Full validation** covers all 9,997 FineWeb and 4,476 Wikipedia sequences via `evaluation.evaluate.run_full_validation`. That's the same `evaluate()` used during training. `train_v3.py` also runs it at the final step.

In [ ]:
if LOG and LOG["val"]:
    print(f"{'step':>7} {'fineweb':>9} {'wikipedia':>10} {'combined':>9} {'ppl':>8}   (SAMPLED)")
    for s, v in sorted(LOG["val"].items())[-10:]:
        c = v.get("combined_weighted", float("nan"))
        print(f"{s:>7} {v.get('fineweb', float('nan')):>9.4f} {v.get('wikipedia', float('nan')):>10.4f} {c:>9.4f} {np.exp(min(c, 20)):>8.1f}")
for s, v in sorted((LOG or {}).get("full_val", {}).items()):
    print(f"FULL validation logged at step {s}: {v}")
if not LOG or not (LOG["val"] or LOG["full_val"]):
    print("No validation results logged yet.")

In [ ]:
RUN_FULL_VALIDATION = False
EVAL_CHECKPOINT = PRODUCTION_CHECKPOINT
EVAL_BATCH_SIZE = 8

FULL_VALIDATION_RESULTS = None
if RUN_FULL_VALIDATION:
    from evaluation.evaluate import run_full_validation
    state = load_compatible_checkpoint(EVAL_CHECKPOINT, require_optimizer=False)
    eval_model = build_model_from_config(config["model"])
    eval_model.load_state_dict(state["model"])
    eval_model.to(DEVICE)
    FULL_VALIDATION_RESULTS = run_full_validation(eval_model, config, DEVICE, EVAL_BATCH_SIZE, NUM_WORKERS)
    FULL_VALIDATION_RESULTS.update(step=int(state["step"]), checkpoint=str(EVAL_CHECKPOINT),
                                   tokenizer_sha256=TOKENIZER_SHA256)
    out = LOG_DIR / f"full_validation_step{state['step']:06d}.json"
    out.write_text(json.dumps(FULL_VALIDATION_RESULTS, indent=2), encoding="utf-8")
    print(json.dumps(FULL_VALIDATION_RESULTS, indent=2))
    print("Saved:", out)
    del state, eval_model
    if HAS_CUDA:
        torch.cuda.empty_cache()
else:
    print("RUN_FULL_VALIDATION is False — set it to True to evaluate", EVAL_CHECKPOINT)

## 13. Checkpoint Management

`train_v3.py` keeps one rolling checkpoint per run. It writes `*.tmp` and then atomically replaces the file, so an interrupted save never corrupts the previous checkpoint. The same file is also the final checkpoint. Periodic copies are made here with `BACKUP_CHECKPOINT`: the copy is verified before it gets its final name, and an existing backup is never overwritten.

In [ ]:
from datetime import datetime

for directory in (CHECKPOINT_DIR, SMOKE_DIR):
    files = sorted(directory.glob("*.pt")) if directory.exists() else []
    print(f"{directory}:")
    for p in files:
        st = p.stat()
        print(f"  {p.name:36s} {st.st_size / 1024**2:9.1f} MB  {datetime.fromtimestamp(st.st_mtime):%Y-%m-%d %H:%M}")
    if not files:
        print("  (no checkpoints)")

In [ ]:
INSPECT_CHECKPOINT = PRODUCTION_CHECKPOINT if PRODUCTION_CHECKPOINT.exists() else SMOKE_DIR / "v3_smoke.pt"

if not INSPECT_CHECKPOINT.exists():
    print("No checkpoint to inspect yet.")
else:
    state = load_compatible_checkpoint(INSPECT_CHECKPOINT)
    weights = state["model"]
    n_params = sum(t.numel() for k, t in weights.items() if "rope" not in k)
    non_finite = [k for k, t in weights.items() if not torch.isfinite(t).all()]
    print(f"Checkpoint      : {INSPECT_CHECKPOINT}")
    print(f"Format / step   : {state['format']} / {state['step']:,}")
    print(f"Architecture    : matches config ({n_params:,} parameters in state dict)")
    print(f"Optimizer state : {len(state['optimizer']['state'])} tensors | scheduler last_epoch {state['scheduler']['last_epoch']}")
    print(f"GradScaler      : {state['scaler'] or 'disabled (CPU run)'}")
    print(f"RNG states      : torch={'torch_rng_state' in state} numpy={'numpy_rng_state' in state} "
          f"python={'python_rng_state' in state} cuda={state.get('cuda_rng_state') is not None}")
    print(f"Saved data paths: {state['config']['data']['train_inputs']}")
    for k in ("embedding.embedding.weight", "layers.0.attention.qkv.weight", "norm.weight"):
        t = weights[k].float()
        print(f"  {k:32s} shape={tuple(t.shape)} mean={t.mean():+.4f} std={t.std():.4f}")
    assert n_params == V3_PARAMETERS and not non_finite, f"Bad weights: {non_finite[:5]}"
    print("Checkpoint integrity PASSED")
    del state, weights

In [ ]:
BACKUP_CHECKPOINT = False   # copy the production checkpoint to a step-tagged file

if BACKUP_CHECKPOINT:
    state = load_compatible_checkpoint(PRODUCTION_CHECKPOINT)
    step = int(state["step"]); del state
    target = CHECKPOINT_DIR / f"100m_v3_step{step:06d}.pt"
    if target.exists():
        raise FileExistsError(f"{target} already exists — not overwriting")
    tmp = target.with_name(target.name + ".tmp")
    shutil.copy2(PRODUCTION_CHECKPOINT, tmp)
    copy_state = load_compatible_checkpoint(tmp)
    if copy_state["step"] != step or tmp.stat().st_size != PRODUCTION_CHECKPOINT.stat().st_size:
        tmp.unlink()
        raise RuntimeError("Backup verification failed; copy removed, original untouched")
    del copy_state
    os.replace(tmp, target)
    print("Verified backup:", target)
else:
    print("BACKUP_CHECKPOINT is False — nothing copied.")

## 14. Text Generation

Uses `inference.generate.generate` with `inference.sampling.sample_next_token`, which supports **temperature** and **top-k**. Top-p and repetition penalty aren't implemented in the project's sampling code. Generation stops at `<EOS>` or after `MAX_NEW_TOKENS`.

In [ ]:
from inference.generate import generate

GEN_CHECKPOINT = PRODUCTION_CHECKPOINT if PRODUCTION_CHECKPOINT.exists() else SMOKE_DIR / "v3_smoke.pt"
PROMPTS = ["Artificial intelligence is", "The capital of France is", "Water is made up of"]
MAX_NEW_TOKENS = 60
TEMPERATURE = 0.7
TOP_K = 50
GEN_SEED = 0

if not GEN_CHECKPOINT.exists():
    print("No checkpoint available for generation yet.")
else:
    state = load_compatible_checkpoint(GEN_CHECKPOINT, require_optimizer=False)
    gen_model = build_model_from_config(config["model"])
    gen_model.load_state_dict(state["model"])
    gen_model.to(DEVICE).eval()
    print(f"Checkpoint {GEN_CHECKPOINT.name} at step {state['step']:,}"
          f"{'  (SMOKE TEST — untrained weights)' if 'smoke' in GEN_CHECKPOINT.name else ''}\n")
    torch.manual_seed(GEN_SEED)
    for prompt in PROMPTS:
        prompt_ids = tokenizer.encode(prompt).ids
        out = generate(gen_model, prompt_ids, MAX_NEW_TOKENS, TEMPERATURE, TOP_K, eos_id=EOS_ID)
        print("PROMPT :", prompt)
        print("OUTPUT :", tokenizer.decode(out[len(prompt_ids):]).strip())
        print("-" * 80)
    del state, gen_model

## 15. Experiment Summary

Collected from this session's measurements, the training log and the checkpoint. Anything not measured is shown as **NOT MEASURED**.

In [ ]:
NM = "NOT MEASURED"
g = globals()
train_rows = LOG["train"] if LOG else []
val = LOG["val"][max(LOG["val"])] if LOG and LOG["val"] else {}
full = FULL_VALIDATION_RESULTS
if full is None:
    saved = sorted(LOG_DIR.glob("full_validation_step*.json"))
    full = json.loads(saved[-1].read_text()) if saved else None
if full is None and LOG and LOG["full_val"]:
    s = max(LOG["full_val"]); full = {f"{k}_loss": v for k, v in LOG["full_val"][s].items()} | {"step": s}
bench = max(g.get("BENCHMARK_RESULTS") or [], key=lambda r: r["batch"], default=None)

summary = [
    ("Parameters", f"{PARAM_COUNT:,}"),
    ("Tokenizer", f"{TOKENIZER_PATH.name} (sha256 {TOKENIZER_SHA256[:12]}…)"),
    ("Training sequences / token positions", f"{DATASET_COUNTS['train']:,} / {TRAIN_TOKEN_POSITIONS:,}"),
    ("Validation sequences (FineWeb / Wikipedia)", f"{DATASET_COUNTS['fineweb_validation']:,} / {DATASET_COUNTS['wikipedia_validation']:,}"),
    ("Planned steps / tokens", f"{config['training']['steps']:,} / {config['training']['steps'] * config['training']['batch_size'] * SEQ_LEN:,}"),
    ("Log analysed", f"{MONITOR_LOG}" if MONITOR_LOG else NM),
    ("Last logged step", f"{train_rows[-1]['step']:,}" if train_rows else NM),
    ("Tokens processed", f"{train_rows[-1]['step'] * config['training']['batch_size'] * SEQ_LEN:,}" if train_rows else NM),
    ("Final train loss (last window)", f"{train_rows[-1]['loss']:.4f}" if train_rows else NM),
    ("Latest sampled val (combined)", f"{val['combined_weighted']:.4f} (ppl {np.exp(min(val['combined_weighted'], 20)):.1f})" if "combined_weighted" in val else NM),
    ("Full val FineWeb / Wikipedia", f"{full['fineweb_loss']:.4f} / {full['wikipedia_loss']:.4f}" if full else NM),
    ("Full val combined (ppl)", f"{full['combined_weighted_loss']:.4f} ({np.exp(min(full['combined_weighted_loss'], 20)):.1f})" if full else NM),
    ("Training duration", NM + " (train_v3.py does not log timestamps)"),
    ("Throughput", f"{bench['tokens_per_s']:,.0f} tokens/s at batch {bench['batch']} (Section 8 benchmark)" if bench else NM),
    ("Hardware", f"{GPU_NAME}" + (f", {GPU_MEMORY_GB:.1f} GB" if HAS_CUDA else "")),
    ("Checkpoint", f"{PRODUCTION_CHECKPOINT}" + ("" if PRODUCTION_CHECKPOINT.exists() else "  (not created yet)")),
]
for name, value in summary:
    print(f"{name:44s} {value}")

## Appendix A — Historical Results (recorded outputs from the previous notebook)

These are **recorded values from earlier runs**, not live measurements. The V2 numbers use the older tokenizer (58 missing ByteLevel characters, ~2% UNK), so they aren't directly comparable to V3.

| Run | Data | Steps | Result |
|---|---|---|---|
| V2 100M (T4, batch 4, FP16) | 738,247 train / 14,790 val sequences (V2 tokenizer) | 20,000 | Full validation loss **4.7366**, perplexity **114.04** |
| V3 smoke test (T4, batch 8) | V3 data | 3 | Train loss 10.53 → 10.54; combined sampled val 10.513 |
| V3 checkpoint resume test | — | 3 | Weights, optimizer (72 tensors), scheduler, GradScaler and RNG states restored exactly |
| V3 memory probe (T4, FP16) | 1 × 512 forward+backward / 8 × 512 full step | 1 | Peak 1.49 GB / 4.29 GB |

V3 data preparation (recorded counts):

| Source | Downloaded | After cleaning | After SHA-256 dedup | Train docs | Val docs | Train sequences | Train tokens |
|---|---:|---:|---:|---:|---:|---:|---:|
| FineWeb-Edu (`default`) | 250,000 | 250,000 | 249,998 | 245,134 | 4,864 | 511,457 | 261,866,097 |
| Wikipedia (`20231101.en`) | 100,000 | 99,729 | 99,717 | 97,720 | 1,997 | 211,028 | 108,046,413 |

The validation split is chosen per document by hashing its text (`sha256(text)[:8] / 0xFFFFFFFF < 0.02`), so a document never lands in both splits.

## Appendix B — Rebuilding the V3 Dataset

The download → clean → dedup → hash-split steps were run inline in the previous Colab notebook (not yet project scripts). Tokenizing the split JSONL files uses the project script:

```bash
python -m data.scripts.tokenize --input <workspace>/splits/fineweb_train.jsonl \
    --tokenizer <workspace>/tokenizer/tokenizer_v3.json \
    --output-dir <workspace>/tokenized_v3/train --block-size 512 --shard-sequences 10000
# repeat for wikipedia_train (→ train) and fineweb_validation / wikipedia_validation (→ validation)
```